In [6]:
import pandas as pd
import sys
sys.path.append('..')

from src.utils import create_player_base, safe_sum, update_rating

In [7]:
all_matches = pd.read_csv('../data/Setka.csv')
player_base = create_player_base(all_matches)

# Standard ELO, K=32 constant 

In [8]:
def process_matches(all_matches, player_base, metric_type="standard", K=32):
    
    results = []

    for match in all_matches.itertuples():
        player_A = match.Player1
        player_B = match.Player2
        r_old_A = player_base.loc[player_base['player'] == player_A, 'rating'].values[0]
        r_old_B = player_base.loc[player_base['player'] == player_B, 'rating'].values[0]
        
        actual_winner = "A" if match.HomeWinner == 1 else "B"
        
        home_points = safe_sum(match.P1_G1, match.P1_G2, match.P1_G3, match.P1_G4, match.P1_G5)
        away_points = safe_sum(match.P2_G1, match.P2_G2, match.P2_G3, match.P2_G4, match.P2_G5)
        
        match_data = {
            "sets_won": match.Sets_P1,
            "sets_lost": match.Sets_P2,
            "point_diff": abs(home_points - away_points)
        }

        r_new_A, r_new_B, E_A, E_B = update_rating(r_old_A, r_old_B, actual_winner, metric_type, match_data, K)
        
        # Record prediction BEFORE updating ratings
        results.append({
            'player1': player_A,
            'player2': player_B,
            'win_probability_p1': E_A,
            'actual_outcome': 1 if actual_winner == "A" else 0
        })
        
        # Now update ratings
        player_base.loc[player_base['player'] == player_A, 'rating'] = r_new_A
        player_base.loc[player_base['player'] == player_B, 'rating'] = r_new_B

    results_df = pd.DataFrame(results)
    results_df.to_csv(f'../results/{metric_type}_K{K}_results.csv', index=False)

In [9]:
process_matches(all_matches, player_base)

TypeError: Invalid value '1500.74' for dtype 'int64'

# Standard ELO, variable K

# MOV Set-based, variable K

# MOV point-based, variable K